This a Notebook file doing some initial Exploratory Data Analysis of the data. 

In [ ]:
import pandas as pd
from IPython.display import display
import plotly.express as px

Below we do some Initial Exploratory Data Analysis with the FRED-MD Data. 

In [172]:
df_raw = pd.read_csv("D:/Users/nickt/ECO482/Research Project Files/Data/2026-01-MD.csv")

In [173]:
df_raw.head()

# Drop first row, as it is just transform codes for the variables
df_raw.drop(df_raw.index[0], inplace=True)

We make a time series, showing the percentage of observations missing for each year/month in the dataset

In [174]:
df_raw['sasdate'] = pd.to_datetime(df_raw['sasdate'])

df_raw['sasdate'] = df_raw['sasdate'].dt.to_period('M')

monthly_missing = df_raw.groupby('sasdate').apply(lambda x: x.isnull().sum().sum() / x.size * 100).reset_index(name='pct_missing')

df_raw['pct_missing'] = df_raw['sasdate'].map(monthly_missing.set_index('sasdate')['pct_missing'])

monthly_missing['sasdate'] = monthly_missing['sasdate'].astype(str)

plt = px.line(monthly_missing, x='sasdate', y='pct_missing', title='Percentage of Missing Values Over Time')
plt.show()

df_raw = df_raw[(df_raw['sasdate'] >= '1970-01-01') & (df_raw['sasdate'] <= '2025-01-01')]

C:\Users\Nick\AppData\Local\Temp\ipykernel_21508\2959559830.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_raw['pct_missing'] = df_raw['sasdate'].map(monthly_missing.set_index('sasdate')['pct_missing'])


We see a decent percentage of observations missing in the late 1950's and early 1960's aswell as a extremely large amount in mid/late 2025. Next we plot the percentage of values miussing for each variable. 

In [175]:
total_missing = (df_raw.isnull().mean() * 100).rename('pct_missing').reset_index()
total_missing = total_missing.rename(columns={'index': 'variable'})
total_missing = total_missing[total_missing['pct_missing'] > 0]

plt = px.bar(
    total_missing,
    x='variable',
    y='pct_missing',
    title='Percent Missing by Variable'
)
plt.show()

We have 3 main variables of concern. As it seems, ACOGNO is unusable as over 40% of that variable is missing. Further Investigation is Required for the rest. 

In [ ]:
##### 'TWEXAFEGSMTHx'
## It seems the first few years (Until Jan 1972) of the TWEXAFEGSMTHx variable are missing, so we will drop the first few years of data to avoid having to impute this variable.
df_raw['TWEXAFEGSMTHx'].head(50)

##### 'UMCSENTx'
## It seems this variable is recorded quarterly until 1977, when it goes to monthly. We will forward fill the missing values for this variable starting at the beginning of the dataset. 
df_raw['UMCSENTx'].head(50)
df_raw['UMCSENTx'].tail(50)

##### 'CP3Mx' & 'COMPAPFFx'
## CP3Mx and COMPAPFFx are missing 1 value in April 2020, so we will forward fill these values for that month.

mask = df_raw[['CP3Mx', 'COMPAPFFx']].isnull().any(axis=1)
df_raw.loc[mask, ['sasdate', 'CP3Mx', 'COMPAPFFx']]

,sasdate,CP3Mx,COMPAPFFx
736,2020-04,NaN,NaN


After adding the suggested changes to the data, we can now load in our cleaned dataset and perform some initial visualizations and summary stats

In [177]:
df = pd.read_csv("D:/Users/nickt/ECO482/Research Project Files/Data/FRED_MD_Cleaned_Transformed.csv")
df_1 = pd.read_csv("D:/Users/nickt/ECO482/Research Project Files/Data/FRED_MD_Cleaned_NonTransformed.csv")

## We see inversions are a rare occurance, only appearing 82 out of 618 Months, or 13.2% of the time.
df['Inversion'].value_counts()

Inversion
0    536
1     82
Name: count, dtype: int64

Below we plot the Slope variable, to see the yield curve inversions over time. All points below the black line are times when the yield curve was inverted. 

In [178]:
plt = px.line(df, x="sasdate", y="Slope", title="Slope of the Yield Curve From Feb 1973 to Jul 2024")
plt.add_hline(y=0)

plt.show()

We can see the correlations between some key variables of our cleaned data (Pre Transformation).

In [179]:
## GS10 is our 10-year treasury yield, 
# TB3MS is our 3-month treasury yield, 
# S&P 500 is our stock market index 
# UNRATE is our unemployment rate
# S&P 500 is our stock market index

cols = ['Slope', 'UNRATE', 'S&P 500']

display(df_1[cols].corr())

,Slope,UNRATE,S&P 500
Slope,1.000000,0.514990,-0.324882
UNRATE,0.514990,1.000000,-0.444495
S&P 500,-0.324882,-0.444495,1.000000
